# Homework

## Setup

Create `uv` project
```bash
uv init
```

Install dependencies
```bash
uv add requests minsearch openai jupyter python-dotenv gitsource
```

## Preparation

In [1]:
import os
from openai import OpenAI
from minsearch import Index
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader

In [4]:
load_dotenv()

False

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level `README`. The lesson pages all live under a module's lessons/ folder, so filtering on `/lessons/` keeps just those.

Each file has a `parse()` method that returns a dictionary with its filename and content:

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

How many lesson pages are in the dataset?

* 24
* 72
* 240
* 720

### Answer

In [7]:
len(documents)

72

> **`72`**

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

### Answer

In [8]:
# Build Minsearch index
index = Index(
    text_fields=["content"],
    keyword_fields=[
        "filename",
    ],
)
index.fit(documents)

In [9]:
# Search
results = index.search("How does the agentic loop keep calling the model until it stops?")
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


> `01-agentic-rag/lessons/14-agentic-loop.md`

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

Call the RAG with the following prompt:

> How does the agentic loop keep calling the model until it stops?

Use `gpt-5.4-mini`. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

In [10]:
from rag_helper import RAGBase
from openai import OpenAI

In [11]:
openai_client = OpenAI()
query = "How does the agentic loop keep calling the model until it stops?"

In [12]:
rag = RAGBase(
    index=index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [14]:
result = rag.rag("How does the agentic loop keep calling the model until it stops?")

In [16]:
print(result.answer)
print(result.usage.input_tokens)

It keeps a `while True` loop around the model call.

Each iteration:
1. sends the full message history to the model,
2. checks the response for any `function_call` items,
3. runs those tools and appends the results to memory,
4. repeats.

It stops when the response has no function calls, i.e. `has_function_calls == False`, then it `break`s out of the loop.
7126


> `7000`

## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

* 70
* 295
* 1100
* 4500

### Answer

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


> `295`

## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer

### Answer

In [20]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

chunk_rag = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    model="gpt-5.4-mini"
)

In [22]:
chunk_response = chunk_rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)
print(chunk_response.answer)
print(chunk_response.usage.input_tokens)

It keeps calling the model inside a `while True` loop, and after each response it checks whether any `function_call` items were returned.

- If there are function calls, the code runs them, appends the results to `messages`, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is: **no function calls returned on the current iteration**.
2309


In [ ]:
response_usage = result.usage.input_tokens
chunk_response_usage = chunk_response.usage.input_tokens

2309


In [27]:
print(f"{round(response_usage/chunk_response_usage, 0)}x fewer tokens")

3.0x fewer tokens


> `3× fewer`

## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20

### Answer

In [29]:
# !uv add toyaikit

In [32]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [33]:
def search(query: str) -> list[dict[str, str]]:
    """Search the course lesson pages for content matching the given query."""
    return chunk_index.search(query, num_results=5)

In [34]:
AGENT_INSTRUCTIONS: str = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

In [35]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [36]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=AGENT_INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [40]:
num_search_calls = sum(
    1
    for msg in result.all_messages
    if hasattr(msg, "type") and msg.type == "function_call" and msg.name == "search"
)
print("search() called:", num_search_calls, "times")

search() called: 3 times


> `4` (closest value)